# Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import cv2
import pydot

# Load and Resize the data

In [2]:
#load_dataset function to load the data and resize the images to 50x50
def load_dataset(directory):
  images = []
  labels = []
  for idx, label in enumerate(uniq_labels):
    for file in os.listdir(directory + '/'+label):
      filepath = directory +'/'+ label + "/" + file
      img = cv2.resize(cv2.imread(filepath),(50,50))
      images.append(img)
      labels.append(idx)
  images = np.asarray(images)
  labels = np.asarray(labels)
  return images, labels

# Display the images

In [3]:
#display_images function to show examples
def display_images(x_data,y_data, title, display_label = True):
    x, y = x_data,y_data
    fig, axes = plt.subplots(5, 8, figsize = (18, 5))
    fig.subplots_adjust(hspace = 0.5, wspace = 0.5)
    fig.suptitle(title, fontsize = 18)
    for i, ax in enumerate(axes.flat):
        ax.imshow(cv2.cvtColor(x[i], cv2.COLOR_BGR2RGB))
        if display_label:
            ax.set_xlabel(uniq_labels[y[i]])
        ax.set_xticks([])
        ax.set_yticks([])
    plt.show()

# Load the data 

In [5]:

# Load all 29 ASL classes
data_dir = r"D:\hand gesture\archive\asl_alphabet_train\asl_alphabet_train"

uniq_labels = sorted(os.listdir(data_dir))

print("Number of classes:", len(uniq_labels))
print("Classes:", uniq_labels)

# Load images and their corresponding labels
X_pre, Y_pre = load_dataset(data_dir)

print("Image array shape:", X_pre.shape)
print("Label array shape:", Y_pre.shape)



Number of classes: 29
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']
Image array shape: (87000, 50, 50, 3)
Label array shape: (87000,)


# Split the data

In [6]:
#spliting dataset into 80% train, 10% validation and 10% test data
X_train, X_test, Y_train, Y_test = train_test_split(X_pre, Y_pre, train_size = 0.8)
X_test, X_eval, Y_test, Y_eval = train_test_split(X_test, Y_test, test_size = 0.5)

In [7]:
%matplotlib notebook

In [8]:
#print shapes and show examples for each set
print("Train images shape",X_train.shape, Y_train.shape)
print("Test images shape",X_test.shape, Y_test.shape)
print("Evaluate image shape",X_eval.shape, Y_eval.shape)
print("Printing the labels",uniq_labels, len(uniq_labels))
display_images(X_train,Y_train,'Samples from Train Set')
display_images(X_test,Y_test,'Samples from Test Set')
display_images(X_eval,Y_eval,'Samples from Validation Set')

Train images shape (69600, 50, 50, 3) (69600,)
Test images shape (8700, 50, 50, 3) (8700,)
Evaluate image shape (8700, 50, 50, 3) (8700,)
Printing the labels ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space'] 29


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Normalize the data 

In [9]:
# converting Y_tes and Y_train to One hot vectors using to_categorical
# example of one hot => '1' is represented as [0. 1. 0. . . . . 0.]
Y_train = to_categorical(Y_train)
Y_test = to_categorical(Y_test)
Y_eval = to_categorical(Y_eval)
X_train = X_train / 255.
X_test = X_test/ 255.
X_eval = X_eval/ 255.

# Build the Model

In [10]:
# building our model
model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(16, (3,3), activation ='relu', input_shape=(50,50,3)),
        tf.keras.layers.Conv2D(16, (3,3), activation ='relu'),
        tf.keras.layers.Conv2D(16, (3,3), activation ='relu'),
        tf.keras.layers.MaxPool2D((2,2)),
        tf.keras.layers.Conv2D(32, (3,3), activation ='relu'),
        tf.keras.layers.Conv2D(32, (3,3), activation ='relu'),
        tf.keras.layers.Conv2D(32, (3,3), activation ='relu'),
        tf.keras.layers.MaxPool2D((2,2)),
        tf.keras.layers.Conv2D(64, (3,3), activation ='relu'),
        tf.keras.layers.Conv2D(64, (3,3), activation ='relu'),
        tf.keras.layers.Conv2D(64, (3,3), activation ='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(29, activation='softmax')
])

model.summary()

d:\hand gesture\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 46, 46, 16)     │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 44, 44, 16)     │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 22, 22, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 20, 20, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 18, 18, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 6, 6, 64)       │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 4, 4, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 2, 2, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 29)             │         3,741 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 157,213 (614.11 KB)

 Trainable params: 157,213 (614.11 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
#compiling the model
#default batch size 32
#default learning rate is 0.001
model.compile(optimizer = 'adam',
               loss = 'categorical_crossentropy',
               metrics=['accuracy'],)

In [12]:
#start training(fitting) the data
history = model.fit(X_train, Y_train, epochs=20, verbose=1,
                validation_data=(X_eval, Y_eval))

Epoch 1/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 109s 47ms/step - accuracy: 0.5624 - loss: 1.4073 - val_accuracy: 0.8826 - val_loss: 0.3411
Epoch 2/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 71s 33ms/step - accuracy: 0.9093 - loss: 0.2636 - val_accuracy: 0.9362 - val_loss: 0.1830
Epoch 3/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 76s 35ms/step - accuracy: 0.9518 - loss: 0.1468 - val_accuracy: 0.9603 - val_loss: 0.1212
Epoch 4/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 105s 48ms/step - accuracy: 0.9656 - loss: 0.1065 - val_accuracy: 0.9749 - val_loss: 0.0776
Epoch 5/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 122s 39ms/step - accuracy: 0.9764 - loss: 0.0779 - val_accuracy: 0.9832 - val_loss: 0.0534
Epoch 6/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 107s 49ms/step - accuracy: 0.9792 - loss: 0.0695 - val_accuracy: 0.9795 - val_loss: 0.0671
Epoch 7/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 123s 56ms/step - accuracy: 0.9811 - loss: 0.0628 - val_accuracy: 0.9870 - val_loss: 0.0407
Epoch 8/20
2175/2175 ━━━━━━━━━━━━━━━━━━━━ 142s 65ms/step - accuracy: 0.

# Save the Model

In [13]:
model.save('hand_gesture_29class.keras')

In [14]:

import os

print("Current working directory:", os.getcwd())
print(
    "Model saved:",
    os.path.exists("hand_gesture_29class.keras")
)


Current working directory: d:\hand gesture\project\hand-gestures
Model saved: True


In [15]:

# Check the saved model and its output classes
import tensorflow as tf
import os

model_path = "hand_gesture_29class.keras"

gesture_model = tf.keras.models.load_model(model_path)

print("Model file exists:", os.path.exists(model_path))
print("Input shape:", gesture_model.input_shape)
print("Output shape:", gesture_model.output_shape)
print("Number of classes:", gesture_model.output_shape[-1])
print("Class labels:", uniq_labels)


Model file exists: True
Input shape: (None, 50, 50, 3)
Output shape: (None, 29)
Number of classes: 29
Class labels: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [16]:

print("Training images minimum:", X_train.min())
print("Training images maximum:", X_train.max())
print("Training images data type:", X_train.dtype)


Training images minimum: 0.0
Training images maximum: 1.0
Training images data type: float64


In [17]:

import matplotlib.pyplot as plt
import numpy as np
import cv2

plt.figure(figsize=(14, 7))

for i in range(8):
    img = X_train[i]

    # OpenCV images are BGR; convert to RGB for display
    img_display = cv2.cvtColor(
        np.clip(img * 255, 0, 255).astype(np.uint8),
        cv2.COLOR_BGR2RGB
    )

    true_index = int(np.argmax(Y_train[i]))
    true_label = uniq_labels[true_index]

    plt.subplot(2, 4, i + 1)
    plt.imshow(img_display)
    plt.title(f"Label: {true_label}")
    plt.axis("off")

plt.tight_layout()
plt.show()


<IPython.core.display.Javascript object>

# Plot the Accuracy & Loss 

In [18]:
train_loss = history.history['loss']
train_acc = history.history['accuracy']
val_loss = history.history['val_loss']
val_accuracy = history.history['val_accuracy']

In [20]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.figure(figsize=(10, 5))
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Confusion Matrix

In [21]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Get the predicted probabilities for each class
Y_pred_probs = model.predict(X_eval)

# Get the predicted class indices
Y_pred = np.argmax(Y_pred_probs, axis=1)

# Compute confusion matrix
conf_matrix = confusion_matrix(np.argmax(Y_eval, axis=1), Y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.show()

272/272 ━━━━━━━━━━━━━━━━━━━━ 7s 21ms/step


<IPython.core.display.Javascript object>